### Importar librerias y configuracion general.

In [ ]:
# =============================================================================
# 1. IMPORTAR LIBRERÍAS Y CONFIGURACIÓN GENERAL
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy as sa
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración (ajusta según tu entorno)
DIRECT_BASE = "C:/Proyectos/Power BI - Logística/Datos generados/"
SERVER = "LAPTOP-OFP910OT"
DATABASE = "Surcomotor_DB"

TABLAS_HECHOS = ["tabla_ventas", "tabla_compras", "tabla_stock_inventario"]
TABLAS_MAESTRAS = ["tabla_productos", "tabla_almacenes", "tabla_clientes"]

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")


### Funcion de Limpieza de Datos y EDA por Categorías

In [ ]:
# =============================================================================
# 2. FUNCIONES DE LIMPIEZA Y EDA POR CATEGORÍA
# =============================================================================

def limpiar_solo_duplicados(df, nombre_tabla):
    """Elimina registros exactamente duplicados y reporta cuántos se quitaron."""
    antes = len(df)
    df_limpia = df.drop_duplicates()
    despues = len(df_limpia)
    if antes > despues:
        print(f"{nombre_tabla}: se eliminaron {antes - despues} duplicados.")
    else:
        print(f"{nombre_tabla}: no se encontraron duplicados.")
    return df_limpia

def eda_tabla_hechos(df, nombre, columna_tipo=None):
    """
    EDA genérico de una tabla de hechos.
    Si se proporciona columna_tipo, se muestran estadísticas separadas
    para cada valor de esa columna.
    """
    print(f"\n{'='*60}")
    print(f"EDA: {nombre.upper()}")
    print(f"{'='*60}")
    print(f"Registros: {len(df):,}")
    print(f"Columnas: {df.shape[1]}")

    # Columnas numéricas excluyendo IDs
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if not c.startswith('id_') and 'ID' not in c]
    cat_cols = df.select_dtypes(include=['object']).columns

    if columna_tipo and columna_tipo in df.columns:
        # EDA por categoría
        for tipo in df[columna_tipo].unique():
            sub = df[df[columna_tipo] == tipo]
            print(f"\n--- Estadísticas para {tipo} ---")
            print(sub[num_cols].describe().round(2))
            for col in num_cols:
                plt.figure(figsize=(8,4))
                sns.histplot(sub[col], kde=True, bins=30)
                plt.title(f'{col} - {tipo}')
                plt.xlabel(col)
                plt.ylabel('Frecuencia')
                plt.show()
    else:
        print("\n--- Estadísticas numéricas generales ---")
        print(df[num_cols].describe().round(2))
        for col in num_cols:
            plt.figure(figsize=(8,4))
            sns.histplot(df[col], kde=True, bins=30)
            plt.title(f'{col}')
            plt.xlabel(col)
            plt.ylabel('Frecuencia')
            plt.show()

    # Top 5 de columnas categóricas
    for col in cat_cols:
        print(f"\n--- Top 5 valores en '{col}' ---")
        print(df[col].value_counts().head(5))

    # Rango de fechas si existe
    if 'Fecha' in df.columns:
        fechas = pd.to_datetime(df['Fecha'])
        print(f"\n--- Rango de fechas ---")
        print(f"Desde: {fechas.min().date()}")
        print(f"Hasta: {fechas.max().date()}")


### Carga y Praparación de Datos

In [ ]:
# =============================================================================
# 3. CARGA Y PREPARACIÓN DE DATOS
# =============================================================================

# Cargar productos para obtener el tipo (vehículo / repuesto)
ruta_productos = os.path.join(DIRECT_BASE, "tabla_productos.csv")
df_prod = pd.read_csv(ruta_productos, encoding='utf-8-sig')
df_prod = limpiar_solo_duplicados(df_prod, "tabla_productos")
# Crear mapeo producto_id -> tipo
producto_tipo = dict(zip(df_prod['ID_PRODUCTO'], df_prod['TIPO']))

# Cargar y limpiar tablas de hechos
for tabla in TABLAS_HECHOS:
    ruta = os.path.join(DIRECT_BASE, f"{tabla}.csv")
    if not os.path.exists(ruta):
        print(f"Archivo no encontrado: {ruta}")
        continue

    df_raw = pd.read_csv(ruta, encoding='utf-8-sig')
    print(f"\nProcesando {tabla}... (original: {len(df_raw)} registros)")
    df_limpia = limpiar_solo_duplicados(df_raw, tabla)

    # Para ventas y compras, agregar columna de tipo de producto
    if tabla == "tabla_ventas" or tabla == "tabla_compras":
        # Unir con productos usando el nombre 'Producto' (coincide con 'DESCRIPCIÓN')
        # Podemos hacer merge con df_prod por descripción o ID.
        # Asumimos que las tablas de hechos tienen columna 'Producto' igual que 'DESCRIPCIÓN'.
        # Si usamos el código generado corregido, 'tabla_compras' y 'tabla_ventas' 
        # contienen 'Producto' igual a la descripción larga.
        # Para ser robustos, mapeamos a través de un diccionario desc -> tipo.
        # Creamos un diccionario a partir de df_prod: descripcion -> tipo
        desc_tipo = dict(zip(df_prod['DESCRIPCIÓN'], df_prod['TIPO']))
        df_limpia['TIPO_PRODUCTO'] = df_limpia['Producto'].map(desc_tipo)
        # Si algún producto no mapea, podríamos asignar un valor, pero no debería ocurrir.
        col_tipo = 'TIPO_PRODUCTO'
    elif tabla == "tabla_stock_inventario":
        # Unir por ID_Producto -> ID_PRODUCTO
        df_limpia = df_limpia.merge(
            df_prod[['ID_PRODUCTO', 'TIPO']],
            left_on='ID_Producto',      # columna en tabla_stock_inventario
            right_on='ID_PRODUCTO',     # columna en tabla_productos
            how='left'
        )
        # Renombrar la columna importada para evitar conflicto con "Tipo" original
        df_limpia.rename(columns={'TIPO': 'TIPO_PRODUCTO'}, inplace=True)
        # Opcional: eliminar la columna de ID duplicada (si ambas aparecen)
        if 'ID_PRODUCTO' in df_limpia.columns and 'ID_Producto' in df_limpia.columns:
            df_limpia.drop(columns=['ID_PRODUCTO'], inplace=True)
        
        col_tipo = 'TIPO_PRODUCTO'

    # EDA segmentado
    eda_tabla_hechos(df_limpia, tabla, columna_tipo=col_tipo)

    # Guardar CSV limpio
    df_limpia.to_csv(os.path.join(DIRECT_BASE, f"{tabla}_clean.csv"),
                     index=False, encoding='utf-8-sig')

    # =========================================================================
    # 4. CARGA A SQL SERVER
    # =========================================================================
    # Inicializar conexión (solo la primera vez)
    if tabla == TABLAS_HECHOS[0]:
        temp_engine = sa.create_engine(
            f"mssql+pyodbc://@{SERVER}/master?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
        )
        with temp_engine.connect() as conn:
            result = conn.execute(sa.text(f"SELECT 1 FROM sys.databases WHERE name = '{DATABASE}'"))
            if not result.fetchone():
                conn.execute(sa.text(f"CREATE DATABASE [{DATABASE}]"))
                print(f"Base de datos '{DATABASE}' creada.")
        temp_engine.dispose()
        engine = sa.create_engine(
            f"mssql+pyodbc://@{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
        )
    else:
        # Reutilizar engine si ya existe, pero por simplicidad lo creamos una vez.
        if 'engine' not in locals():
            engine = sa.create_engine(
                f"mssql+pyodbc://@{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
            )

    df_limpia.to_sql(tabla, engine, if_exists='replace', index=False)
    print(f"✅ Tabla '{tabla}' cargada en SQL Server ({len(df_limpia)} registros).")

# Cargar y limpiar tablas maestras
for tabla in TABLAS_MAESTRAS:
    ruta = os.path.join(DIRECT_BASE, f"{tabla}.csv")
    if not os.path.exists(ruta):
        print(f"Archivo no encontrado: {ruta}")
        continue
    df = pd.read_csv(ruta, encoding='utf-8-sig')
    df = limpiar_solo_duplicados(df, tabla)
    df.to_csv(os.path.join(DIRECT_BASE, f"{tabla}_clean.csv"),
              index=False, encoding='utf-8-sig')
    df.to_sql(tabla, engine, if_exists='replace', index=False)
    print(f"✅ Tabla maestra '{tabla}' cargada con {len(df)} registros.")

print("\n🎉 Proceso completo. Todas las tablas fueron reemplazadas en la base de datos.")
